# 06 — Lựa chọn và giảm chiều đặc trưng

**Mục tiêu:** tìm số lượng và tập đặc trưng phù hợp theo 3 tầng: lọc thống kê, lọc theo mô hình, giảm chiều.

**Rubric:** Bảng — lựa chọn đặc trưng kèm CV F1 (10%).

**Đề yêu cầu:**
- [ ] Lọc thống kê: ANOVA F-test, Chi-square, Mutual Information.
- [ ] Lọc theo mô hình: feature importance từ Random Forest và Gradient Boosting; RFE có cross-validation.
- [ ] Giảm chiều: PCA, t-SNE (trực quan), UMAP nếu dữ liệu lớn.
- [ ] Với mỗi phương pháp lọc: huấn luyện mô hình, báo cáo F1 5-fold CV, vẽ biểu đồ hiệu năng theo số đặc trưng.

**Đầu vào:** development set qua các bước đã chọn ở 03–05.

**Đầu ra:** `results/tabular/tables/feature_selection_cv.csv`, biểu đồ F1 theo số đặc trưng, danh sách đặc trưng được chọn.
---
*Quy ước khung:* ô **[Code]** sẽ được thay bằng ô code; ô **[Nhận xét]** chỉ viết sau khi có kết quả thật. Mọi score/p-value phải được **in ra và gán vào biến Python** (yêu cầu của đề).

## 0. Thiết lập

> **[Code]** Import thư viện; tìm gốc repo (`find_project_root`, nên chuyển vào `src/data_utils.py`); đọc `data/raw/home_credit/application_train.csv`; nạp `data/splits/home_credit_split_ids.csv` và **chỉ lấy phần development**. Không dùng phần test trong notebook này. `SEED = 42`. Đặt bước chọn đặc trưng **bên trong** Pipeline để mỗi fold tự chọn lại (chống leakage). Lưới số đặc trưng, ví dụ k ∈ {5, 10, 20, 40, 80, tất cả}.

> **Lưu ý về F1:** baseline ở notebook 01 có recall lớp 1 chỉ ≈ 0,011 với ngưỡng 0,5, nên F1 lớp 1 gần 0 và khó so sánh. Cần ghi rõ đang dùng F1 lớp 1 hay F1-macro, và có dùng `class_weight="balanced"` cho mô hình đánh giá hay không — giữ cố định cho mọi phương pháp.

## 1. ANOVA F-test

**Lý thuyết — cần trình bày:**
- $F = \frac{MS_{between}}{MS_{within}}$ (phương sai giữa các nhóm / phương sai trong nhóm) cho từng đặc trưng số theo nhãn.

> **[Code]** `SelectKBest(f_classif)` với lưới k; F1 5-fold CV cho từng k.

> **[Nhận xét]** Những đặc trưng nào có F cao nhất? Có trùng với tương quan ở notebook 02 không?

## 2. Chi-square

**Lý thuyết — cần trình bày:**
- $\chi^2 = \sum \frac{(O - E)^2}{E}$; yêu cầu đặc trưng **không âm** (đếm/One-Hot/Min-Max).

> **[Code]** `SelectKBest(chi2)` trên dữ liệu không âm; F1 5-fold CV theo k.

> **[Nhận xét]** Chi-square phù hợp với loại đặc trưng nào?

## 3. Mutual Information

**Lý thuyết — cần trình bày:**
- $I(X;Y) = \sum p(x,y)\log\frac{p(x,y)}{p(x)p(y)}$ — bắt được quan hệ phi tuyến; ước lượng bằng kNN nên tốn thời gian.

> **[Code]** `SelectKBest(mutual_info_classif)` (lấy mẫu nếu cần, ghi rõ); F1 5-fold CV theo k.

> **[Nhận xét]** MI chọn đặc trưng khác ANOVA ở đâu?

## 4. Importance từ Random Forest và Gradient Boosting

**Lý thuyết — cần trình bày:**
- Impurity-based importance và thiên lệch về biến nhiều giá trị; permutation importance là cách kiểm tra lại.

> **[Code]** Huấn luyện RF và GB, xếp hạng importance, chọn top-k theo lưới; F1 5-fold CV.

> **[Nhận xét]** Hai mô hình có đồng thuận về đặc trưng quan trọng không?

## 5. RFE có cross-validation

**Lý thuyết — cần trình bày:**
- Loại dần đặc trưng kém quan trọng nhất, dùng CV để chọn số đặc trưng.

> **[Code]** `RFECV` với một estimator nhanh (ví dụ Logistic Regression), `scoring` F1 đã chọn; vẽ điểm theo số đặc trưng.

> **[Nhận xét]** Số đặc trưng tối ưu là bao nhiêu? Đường điểm có phẳng sau một ngưỡng không?

## 6. So sánh các phương pháp lọc

**Lý thuyết — cần trình bày:**
- Tiêu chí: F1 CV, số đặc trưng, độ ổn định giữa các fold, thời gian.

> **[Code]** Một biểu đồ F1 theo số đặc trưng cho mọi phương pháp; bảng tổng hợp lưu vào biến `fs_results`; độ chồng lấp các tập đặc trưng được chọn.

> **[Nhận xét]** Chọn phương pháp và số đặc trưng cho pipeline cuối.

## 7. Giảm chiều: PCA

**Lý thuyết — cần trình bày:**
- Thành phần chính là hướng phương sai lớn nhất; scree plot; số thành phần cho 90%/95% phương sai.
- Giữ phương sai **không đồng nghĩa** giữ thông tin dự đoán nhãn.

> **[Code]** PCA trên dữ liệu đã chuẩn hóa; scree plot; F1 CV khi dùng k thành phần.

> **[Nhận xét]** Cần bao nhiêu thành phần? PCA có giúp mô hình không?

## 8. Trực quan: t-SNE (và UMAP)

**Lý thuyết — cần trình bày:**
- t-SNE giữ cấu trúc lân cận, chỉ để quan sát; khoảng cách giữa cụm không có nghĩa định lượng; tham số perplexity.
- UMAP: thư viện `umap-learn` **không có trong danh sách thư viện của đề** → hỏi giảng viên trước khi dùng.

> **[Code]** t-SNE trên mẫu (ghi số dòng, seed), tô màu theo TARGET.

> **[Nhận xét]** Hai lớp có tách nhau trong không gian 2D không? Không dùng t-SNE để kết luận về khả năng phân loại.

## Tổng kết và lựa chọn

> **[Nhận xét]** Tập đặc trưng / phương pháp giảm chiều được chọn cho pipeline cuối.

## Hạn chế

> **[Nhận xét]** Chi phí tính toán, kết quả trên mẫu, phụ thuộc vào mô hình đánh giá.